# Tech Challenge - Fase 1: Case NPS Preditivo
**Objetivo:** Transformar dados operacionais em insights estratégicos para prever e melhorar a satisfação do cliente (NPS).

Este projeto segue a metodologia **CRISP-DM** e as boas práticas de código **PEP 8**.

## 1. Entendimento do Negócio (Business Understanding)
A empresa enfrenta alta variabilidade no NPS. Atualmente, a medição é reativa: o score é coletado apenas após a jornada de compra, o que impede ações preventivas.

**Problema de negócio:** identificar quais fatores operacionais impactam a satisfação do cliente e antecipar clientes propensos a se tornar detratores.

**Por que o NPS é importante para um e-commerce?**
- NPS mensura a experiência do cliente de forma direta.
- Permite detectar falhas recorrentes em logística e atendimento.
- Ajuda a priorizar ações que aumentam retenção, receita e reputação.

**Áreas beneficiadas:** logística, atendimento ao cliente, produto, pricing, customer success e operação.

**Impacto do NPS em negócio:**
- Recompra: clientes promotores são mais propensos a comprar novamente.
- Boca a boca: clientes satisfeitos recomendam e atraem novos consumidores.
- Market share: melhores experiências aumentam diferenciação competitiva.

**Indicadores externos úteis:** benchmarks de NPS do setor, SLA logístico, prazo de entrega, taxa de devolução e análise da concorrência.

In [ ]:
# Importação de bibliotecas (Padrão PEP 8)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Configurações de visualização
plt.style.use('default')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 14

In [ ]:
# Carregamento dos dados
df = pd.read_csv('../data/desafio_nps_fase_1.csv')

# Limpeza inicial: Removendo IDs (sem valor preditivo)
df_clean = df.drop(columns=['customer_id', 'order_id'])

# Feature Engineering: Categorização de NPS e variável-alvo para predição de detratores
df_clean['nps_category'] = pd.cut(
    df_clean['nps_score'],
    bins=[-0.1, 6, 8, 10],
    labels=['Detrator', 'Passivo', 'Promotor'],
)
df_clean['is_detractor'] = (df_clean['nps_score'] <= 6).astype(int)

print('Shape do dataset:', df_clean.shape)
df_clean.head()

## 2. Definição da Target (Target Definition)
**Variável de satisfação:** `nps_score` representa a nota de satisfação do cliente medida de 0 a 10 após a compra.

**Alvo de negócio:** prever clientes detratores antes da aplicação da pesquisa, usando os dados operacionais disponíveis.

**Por que essa variável foi escolhida?**
- É o indicador direto de satisfação usado pelo desafio.
- Já está coletada no momento final da jornada e permite rotular exemplos históricos.

**Momento de coleta:** após o encerramento da experiência de compra.

**Riscos de uso inadequado:**
- Usar `nps_score` como variável de entrada para predição sem garantir que ela seja posterior às decisões operacionais.
- Confundir correlação com causalidade ao propor ações apenas com base em variáveis correlacionadas.

## 3. Análise Exploratória de Dados (EDA)
Vamos investigar quais variáveis têm maior correlação com a nota final e caracterizar o volume de detratores.

In [ ]:
# Distribuição das categorias de NPS
plt.figure(figsize=(8, 4))
sns.countplot(data=df_clean, x='nps_category', color='#4A90E2')
plt.title('Distribuição de NPS por Categoria')
plt.xlabel('Categoria NPS')
plt.ylabel('Contagem')
plt.show()

# Distribuição das notas de NPS
plt.figure(figsize=(8, 4))
sns.histplot(df_clean['nps_score'], bins=11, kde=False, color='#4A90E2')
plt.title('Distribuição do NPS Score')
plt.xlabel('NPS Score')
plt.ylabel('Contagem')
plt.show()

# Taxa de detratores por região
region_detractor_rate = df_clean.groupby('customer_region')['is_detractor'].mean().sort_values(ascending=False)
plt.figure(figsize=(8, 4))
sns.barplot(x=region_detractor_rate.index, y=region_detractor_rate.values, color='#E76F51')
plt.title('Taxa de Detratores por Região')
plt.xlabel('Região do Cliente')
plt.ylabel('Proporção de Detratores')
plt.ylim(0, 1)
plt.show()

# Matriz de correlação
plt.figure(figsize=(12, 8))
df_num = df_clean.select_dtypes(include=[np.number])
sns.heatmap(df_num.corr(), annot=True, cmap='RdBu', fmt='.2f', center=0)
plt.title('Matriz de Correlação das Variáveis Operacionais')
plt.show()

In [ ]:
# Gráficos para identificar os principais motivos do NPS baixo
# Atraso na entrega
delay_labels = ['No delay', '1 dia', '2 dias', '3 dias', '4-10 dias', '>10 dias']
delay_bins = [-1, 0, 1, 2, 3, 10, df_clean['delivery_delay_days'].max() + 1]
df_clean['delay_group'] = pd.cut(df_clean['delivery_delay_days'], bins=delay_bins, labels=delay_labels)

plt.figure(figsize=(10, 5))
sns.boxplot(data=df_clean, x='delay_group', y='nps_score', color='#4A90E2')
plt.title('NPS Score por Faixa de Atraso na Entrega')
plt.xlabel('Faixa de Atraso')
plt.ylabel('NPS Score')
plt.show()

# Reclamações e atendimento
complaints_labels = ['0', '1', '2', '3', '4+']
complaints_bins = [-1, 0, 1, 2, 3, df_clean['complaints_count'].max() + 1]
df_clean['complaints_group'] = pd.cut(df_clean['complaints_count'], bins=complaints_bins, labels=complaints_labels)

plt.figure(figsize=(10, 5))
sns.boxplot(data=df_clean, x='complaints_group', y='nps_score', color='#E63946')
plt.title('NPS Score por Número de Reclamações')
plt.xlabel('Reclamações Registradas')
plt.ylabel('NPS Score')
plt.show()

plt.figure(figsize=(10, 5))
sns.boxplot(data=df_clean, x='customer_service_contacts', y='nps_score', color='#F4A261')
plt.title('NPS Score por Número de Contatos com SAC')
plt.xlabel('Contatos com SAC')
plt.ylabel('NPS Score')
plt.show()

plt.figure(figsize=(10, 5))
sns.scatterplot(data=df_clean, x='csat_internal_score', y='nps_score', hue='is_detractor', palette=['#2A9D8F', '#E76F51'], alpha=0.7)
plt.title('Relação entre CSAT Interno e NPS')
plt.xlabel('CSAT Interno')
plt.ylabel('NPS Score')
plt.legend(title='Detrator', labels=['Não', 'Sim'])
plt.show()

summary_delay = df_clean.groupby('delay_group')['nps_score'].agg(['mean', 'count'])
print(summary_delay)

## 4. Por que o NPS está baixo?
Os gráficos acima mostram que os principais motivos do NPS baixo são:
- atrasos na entrega, que reduzem substancialmente a nota média;
- aumento no número de reclamações e de contatos com o SAC;
- relações mais fracas entre CSAT interno e NPS quando o cliente é detrator.

Esses insights serão úteis para destacar visualmente na apresentação as causas do NPS baixo.

## 5. Análise Estatística (Rigor Científico)
Validando se as diferenças observadas são estatisticamente significativas entre entregas no prazo e com atraso.

In [ ]:
# Teste de Hipótese: Grupos com Atraso vs. No Prazo
grupo_atraso = df_clean[df_clean['delivery_delay_days'] > 0]['nps_score']
grupo_prazo = df_clean[df_clean['delivery_delay_days'] == 0]['nps_score']

t_stat, p_val = stats.ttest_ind(grupo_atraso, grupo_prazo, equal_var=False)
print(f"Estatística T: {t_stat:.4f} | P-valor: {p_val:.4e}")
if p_val < 0.05:
    print("Resultado: Existe uma diferença estatisticamente significativa entre os grupos.")
else:
    print("Resultado: Não há evidência de diferença significativa.")

## 6. Modelagem e Importância de Variáveis
Construindo um modelo de classificação para prever detratores antes da pesquisa de NPS ser respondida.

In [ ]:
# Preparação dos dados para classificação
feature_columns = df_clean.drop(columns=['nps_score', 'nps_category', 'is_detractor']).columns.tolist()
X = df_clean[feature_columns]
y = df_clean['is_detractor']

numeric_columns = X.select_dtypes(include=[np.number]).columns.tolist()
category_columns = X.columns.difference(numeric_columns).tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), category_columns),
    ]
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)),
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print('Accuracy:', accuracy_score(y_test, y_pred))
print('ROC AUC:', roc_auc_score(y_test, y_prob))
print('\nClassification report:\n', classification_report(y_test, y_pred, target_names=['Não Detrator', 'Detrator']))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Não Detrator', 'Detrator'], yticklabels=['Não Detrator', 'Detrator'])
plt.xlabel('Predito')
plt.ylabel('Real')
plt.title('Matriz de Confusão')
plt.show()

In [ ]:
# Importância das características do modelo
classifier = model.named_steps['classifier']
encoded_features = (
    numeric_columns +
    model.named_steps['preprocessor']
    .named_transformers_['cat']
    .get_feature_names_out(category_columns)
    .tolist()
)
feature_importance = pd.Series(classifier.coef_[0], index=encoded_features).sort_values()
feature_importance.plot(kind='barh', color='teal')
plt.title('Impacto das Variáveis na Predição de Detratores')
plt.xlabel('Coeficiente do Modelo')
plt.show()

## 7. Storytelling e Conclusões Executivas
**Principais Insights:**
1. **Atraso na entrega** é um fator central na redução de NPS e na probabilidade de um cliente ser detrator.
2. **Número de contatos com SAC** também é um sinal importante de maior insatisfação.
3. **Predição de detratores** antes do envio da pesquisa é plausível com os dados operacionais disponíveis.

**Recomendações:**
- Monitorar e priorizar pedidos com atraso maior que 2 dias.
- Investir em resolução no primeiro contato e atendimento proativo.
- Construir um dashboard de risco de detrator para ações imediatas do time de customer success.